# つぎミー 発表用試算（2026-09-26）
## 結論
企業が給与とは別に利用料を支払う方針。時給1,300円、月64時間、料率30%を検証案とする。給与83,200円、利用料24,960円。変動費後10,960円、獲得・初期設定84,000円の回収7.7稼働月。すべて事業計画上の仮定で、販売・継続実績ではない。
## 方法と出典
候補者1人・1社を1組として月単位で計算。税別の利用料。交通費、保険等の企業負担、消費税は企業小計に含まない。給与はつぎミーの売上に含めない。24か月継続・月64時間は計画基準。
- 方針：[Notion](https://app.notion.com/p/3e6729af3b7d80618787efdb74bf08c6)
- [大阪最低賃金](https://jsite.mhlw.go.jp/osaka-roudoukyoku/jirei_toukei/saitei_chingin/saitei.html)：2026/10/1から1,231円。時給1,300円は提案値。
- [タイミー](https://go.timee.co.jp/l/1019862/2023-05-11/b5kt)：日当・交通費の30%。対象サービスが異なり、支払い意向の根拠ではない。
- [職業紹介](https://jsite.mhlw.go.jp/osaka-roudoukyoku/hourei_seido_tetsuzuki/yuryou_muryou_shokugyou.html)：運営体制・対象業務は実証前に確認。
## 計算
少数の仮定の監査用なので、表と計算値を用いる。市場実測データ・予測モデルではない。

In [1]:
import math, json
from IPython.display import display, HTML
hourly_wage=1300
hours=64
rate=.30
support_hours=3
hourly_cost=3000
system=2000
reserve=3000
acquisition=60000
onboarding=24000
fixed_lean=100000
fixed_full=400000
setup=300000
wage=hourly_wage*hours
revenue=round(wage*rate)
variable=support_hours*hourly_cost+system+reserve
contribution=revenue-variable
upfront=acquisition+onboarding
def table(headers, rows):
    import html
    display(HTML('<table><tr>'+''.join('<th>'+html.escape(str(h))+'</th>' for h in headers)+'</tr>'+''.join('<tr>'+''.join('<td>'+html.escape(str(x))+'</td>' for x in row)+'</tr>' for row in rows)+'</table>'))
table(['項目','円/月 または月数'],[['給与',wage],['利用料',revenue],['企業負担小計（諸費用別）',wage+revenue],['変動費',variable],['固定費前残額',contribution],['初期投資',upfront],['回収稼働月',round(upfront/contribution,2)]])
assert wage==83200 and revenue==24960 and variable==14000 and contribution==10960
assert contribution*24-upfront==179040


項目,円/月 または月数
給与,83200
利用料,24960
企業負担小計（諸費用別）,108160
変動費,14000
固定費前残額,10960
初期投資,84000
回収稼働月,7.66


### 感度分析
一度に一つの条件を変える。月32時間でも支援費が一定なら赤字月になる。

In [2]:
table(['料率','月売上','変動費後残額','初期回収月'], [[r,round(wage*r),round(wage*r)-variable,round(upfront/(round(wage*r)-variable),2)] for r in [.2,.3,.4]])
table(['月間時間','月売上','変動費後残額'], [[h,round(hourly_wage*h*rate),round(hourly_wage*h*rate)-variable] for h in [32,64,96]])


料率,月売上,変動費後残額,初期回収月
0.2,16640,2640,31.82
0.3,24960,10960,7.66
0.4,33280,19280,4.36


月間時間,月売上,変動費後残額
32,12480,-1520
64,24960,10960
96,37440,23440


### 初年度と定常状態
初年度準備2か月、その後毎月1組開始。離脱なし。固定報酬なし案も変動の営業・支援人件費は計上済み。資金枠は年次費用の仮定で、月別現金残高を保証しない。

In [3]:
active=[0,0,1,2,3,4,5,6,7,8,9,10]
person_months=sum(active)
starts=10
annual_revenue=person_months*revenue
annual_variable=person_months*variable
annual_upfront=starts*upfront
rows=[]
for label,fixed in [('創業者固定報酬なし',fixed_lean),('固定報酬等を月30万円追加',fixed_full)]:
    result=annual_revenue-annual_variable-annual_upfront-fixed*12
    budget=math.ceil((-result+setup)*1.3/100000)*100000
    rows.append([label,annual_revenue,annual_variable,annual_upfront,fixed*12,result,budget])
table(['案','売上','変動費','獲得設定','固定費','差額','予備含む計画枠'],rows)
assert person_months==55 and annual_revenue==1372800
assert sum((revenue-variable)*a for a in active)-840000-1200000==-1437200
breakeven=math.ceil(fixed_full/(contribution-upfront/24))
print('定常状態の必要組数:',breakeven,'月業務時間:',breakeven*3+breakeven/24*28)
assert breakeven==54
print('稼働量20%減時:',44*revenue,44*contribution-840000-fixed_lean*12)
results={'monthly_revenue':revenue,'monthly_contribution':contribution,'payback_months':upfront/contribution,'annual_revenue':annual_revenue,'annual_cases':rows,'steady_breakeven':breakeven}


案,売上,変動費,獲得設定,固定費,差額,予備含む計画枠
創業者固定報酬なし,1372800,770000,840000,1200000,-1437200,2300000
固定報酬等を月30万円追加,1372800,770000,840000,4800000,-5037200,7000000


定常状態の必要組数: 54 月業務時間: 225.0
稼働量20%減時: 1098240 -1557760


### 譲渡代金の例
契約推奨や企業価値評価ではない。売上連動3%、上限500万円の支払い能力を比較する。利益と現金は別。売り手の同意と専門家の確認が必要。

In [4]:
table(['年商','年支払額(3%)','500万円に届く単純年数','7年の支払累計(上限500万円)'],[[s,round(s*.03),round(5000000/(s*.03),2),min(5000000,round(s*.03*7))] for s in [30000000,20000000]])
assert round(20000000*.03*7)==4200000
print('旧1500万円例:',round(15000000/(70000000*.03),2),'年、営業利益500万円に対する年支払い割合:',(70000000*.03)/5000000)


年商,年支払額(3%),500万円に届く単純年数,7年の支払累計(上限500万円)
30000000,900000,5.56,5000000
20000000,600000,8.33,4200000


旧1500万円例: 7.14 年、営業利益500万円に対する年支払い割合: 0.42


## 判断
30%を最初の料金提示案にする。20%では本仮定の24か月でも初期費用を回収できない。最初の有料1社から工数・獲得費・継続意思を記録する。54組は固定費等を賄う条件計算で、到達予測ではない。初年度売上に補助金・成約手数料を入れない。
準備費・提携費は見積もり前で、職業紹介の財産要件や専門家費用を網羅していない。法令対応、譲渡代金、卒業後勤務条件が事業化前の主要な残課題。
